# Machine Learning — Lab 8
## Decision Trees: Splitting, Entropy, and Pruning

**Main Course Learning Outcomes — CLO3, CLO4, CLO5**

- **CLO3:** Construct and analyze machine learning models by applying algorithmic principles and internal computations.
- **CLO4:** Apply supervised learning techniques to solve practical problems and interpret their outcomes.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** A tree that reaches high accuracy is not enough. Full credit requires you to explain **why a split was chosen, how impurity changed, how depth affects generalization, and why pruning or stopping rules may improve validation performance**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Tree concepts & manual impurity | 20 min | Compute entropy, Gini, and information gain |
| 2. Manual root-split selection | 25 min | Compare candidate splits and choose the best root |
| 3. Train a decision tree | 20 min | Fit a classifier and interpret the learned structure |
| 4. Depth & overfitting experiment | 20 min | Compare training and validation performance across depths |
| 5. Pruning / stopping controls | 20 min | Study `max_depth`, `min_samples_leaf`, and cost-complexity pruning |
| 6. Challenge, debugging & viva | 15 min | Defend a split, diagnose leakage/overfitting, explain one prediction |
| **Total** | **120 min** | |

### Main idea

$$
\boxed{
\text{Mixed node}
\rightarrow
\text{candidate splits}
\rightarrow
\text{impurity reduction}
\rightarrow
\text{recursive tree}
\rightarrow
\text{complexity control}
}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. explain root nodes, internal nodes, branches, leaves, and depth;
2. compute entropy for a binary classification node;
3. compute Gini impurity;
4. compute weighted child impurity and information gain;
5. select the best split from small candidate splits;
6. fit a decision tree using `scikit-learn`;
7. trace one prediction from root to leaf;
8. explain how tree depth affects bias and variance;
9. use validation data to tune tree complexity;
10. explain pre-pruning and post-pruning conceptually.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import (
    DecisionTreeClassifier,
    plot_tree,
    export_text
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 150)

print("Machine Learning Lab 8 environment ready.")

# Part I — Manual Tree Mechanics

A decision tree chooses a split that makes the child nodes **purer** than the parent node.

For a binary node with class probabilities $p_+$ and $p_-$:

### Entropy

$$
H(S)
=
-p_+\log_2(p_+)
-p_-\log_2(p_-)
$$

### Gini impurity

$$
Gini(S)
=
1-p_+^2-p_-^2.
$$

A pure node has impurity 0.

## Task 1.1 — Implement Entropy

Complete the function.

The function receives counts for two classes.

In [ ]:
def binary_entropy(n_positive, n_negative):
    total = n_positive + n_negative

    if total == 0:
        return 0.0

    probs = np.array([
        n_positive / total,
        n_negative / total
    ], dtype=float)

    # TODO: remove zero probabilities, then compute entropy.
    nonzero = None
    entropy = None

    return float(entropy)

In [ ]:
# Self-check
assert abs(binary_entropy(5, 5) - 1.0) < 1e-12
assert abs(binary_entropy(10, 0) - 0.0) < 1e-12
assert abs(binary_entropy(8, 2) - 0.7219280949) < 1e-8

print("Entropy function tests passed.")

## Task 1.2 — Implement Gini Impurity

In [ ]:
def binary_gini(n_positive, n_negative):
    total = n_positive + n_negative

    if total == 0:
        return 0.0

    p_pos = n_positive / total
    p_neg = n_negative / total

    # TODO: implement Gini impurity.
    gini = None

    return float(gini)

In [ ]:
assert abs(binary_gini(5, 5) - 0.5) < 1e-12
assert abs(binary_gini(10, 0) - 0.0) < 1e-12
assert abs(binary_gini(8, 2) - 0.32) < 1e-12

print("Gini function tests passed.")

## Task 1.3 — Predict Before Calculating

Rank these nodes from **most pure** to **most impure**.

| Node | Positive | Negative |
|---|---:|---:|
| A | 10 | 0 |
| B | 8 | 2 |
| C | 5 | 5 |
| D | 1 | 9 |

Then predict which node has the largest entropy.

**Your prediction:**

In [ ]:
nodes = pd.DataFrame({
    "node": ["A", "B", "C", "D"],
    "positive": [10, 8, 5, 1],
    "negative": [0, 2, 5, 9],
})

nodes["entropy"] = [
    binary_entropy(p, n)
    for p, n in zip(nodes["positive"], nodes["negative"])
]

nodes["gini"] = [
    binary_gini(p, n)
    for p, n in zip(nodes["positive"], nodes["negative"])
]

display(nodes.round(4))

## Task 1.4 — Interpret Impurity

Answer:

1. Which node is pure?
2. Which node has maximum binary entropy?
3. Why do nodes B and D have the same impurity?
4. What does that tell you about impurity measures?

# Part II — Information Gain

Suppose a parent node $S$ is split into child nodes $S_1,\ldots,S_m$.

The weighted child entropy is:

$$
H_{children}
=
\sum_j
\frac{|S_j|}{|S|}
H(S_j).
$$

Information gain is:

$$
IG
=
H(S)-H_{children}.
$$

A larger information gain means the split reduced uncertainty more strongly.

## Task 2.1 — Implement Weighted Entropy

Each child is represented as:

```text
(positive_count, negative_count)
```

Complete:

In [ ]:
def weighted_child_entropy(children):
    totals = np.array(
        [p + n for p, n in children],
        dtype=float
    )

    grand_total = totals.sum()

    # TODO: compute weighted average entropy.
    weighted = None

    return float(weighted)

In [ ]:
children_check = [(4, 1), (1, 4)]
value = weighted_child_entropy(children_check)

expected = 0.5 * binary_entropy(4, 1) + 0.5 * binary_entropy(1, 4)
assert abs(value - expected) < 1e-12

print("Weighted entropy test passed.")

## Task 2.2 — Implement Information Gain

In [ ]:
def information_gain(parent_counts, children):
    p_parent, n_parent = parent_counts

    # TODO:
    # 1. compute parent entropy
    # 2. compute weighted child entropy
    # 3. subtract
    parent_entropy = None
    children_entropy = None
    gain = None

    return float(gain)

In [ ]:
gain_check = information_gain(
    (5, 5),
    [(4, 1), (1, 4)]
)

assert gain_check > 0
assert gain_check < 1

print("Information-gain test passed.")
print("Example gain:", round(gain_check, 4))

## Task 2.3 — Manual Root Split

Consider this small dataset.

Target: `At_Risk`

- `1` = at risk
- `0` = not at risk

| Student | Attendance | Missed Labs | At_Risk |
|---|---|---|---:|
| S1 | High | Low | 0 |
| S2 | High | Low | 0 |
| S3 | High | High | 1 |
| S4 | High | High | 0 |
| S5 | Low | Low | 1 |
| S6 | Low | Low | 1 |
| S7 | Low | High | 1 |
| S8 | Low | High | 1 |

### Candidate Split A — Attendance

- High: 1 positive, 3 negative
- Low: 4 positive, 0 negative

### Candidate Split B — Missed Labs

- Low: 2 positive, 2 negative
- High: 3 positive, 1 negative

Before running code, predict which split has larger information gain.

In [ ]:
parent = (5, 3)

gain_attendance = information_gain(
    parent,
    [(1, 3), (4, 0)]
)

gain_missed_labs = information_gain(
    parent,
    [(2, 2), (3, 1)]
)

print("Parent entropy:", round(binary_entropy(*parent), 4))
print("Information gain — Attendance:", round(gain_attendance, 4))
print("Information gain — Missed Labs:", round(gain_missed_labs, 4))

## Task 2.4 — Explain the Root Choice

Answer:

1. Which split should be the root?
2. Why?
3. Which branch becomes immediately pure?
4. Why is tree construction described as **greedy**?
5. Does choosing the best local split guarantee the globally best possible tree?

# Part III — Build a Real Decision Tree

We now use a synthetic **student success-risk** dataset.

Target:

$$
y=1 \quad \text{At Risk}
$$

Features:

- `attendance_pct`
- `quiz_score`
- `missed_labs`
- `lms_activity`
- `previous_gpa`
- `commute_minutes`

The data contain nonlinear interactions so a tree can capture useful rules.

In [ ]:
rng = np.random.default_rng(8120)
n = 900

attendance = np.clip(rng.normal(81, 12, n), 35, 100)
quiz = np.clip(rng.normal(72, 14, n), 20, 100)
missed_labs = np.clip(rng.poisson(2.2, n), 0, 8)
lms_activity = np.clip(rng.gamma(2.2, 18, n), 0, 140)
previous_gpa = np.clip(rng.normal(2.9, 0.55, n), 1.2, 4.0)
commute = np.clip(rng.gamma(2.0, 12, n), 2, 90)

# Nonlinear risk structure.
risk_score = (
    1.8 * (attendance < 70)
    + 1.5 * (quiz < 60)
    + 1.2 * (missed_labs >= 4)
    + 1.0 * ((attendance < 78) & (quiz < 68))
    + 0.9 * (previous_gpa < 2.4)
    + 0.5 * (lms_activity < 20)
    + 0.35 * (commute > 50)
    + rng.normal(0, 0.8, n)
)

risk_prob = 1 / (1 + np.exp(-(risk_score - 1.6)))
at_risk = rng.binomial(1, risk_prob)

df_master = pd.DataFrame({
    "attendance_pct": np.round(attendance, 1),
    "quiz_score": np.round(quiz, 1),
    "missed_labs": missed_labs,
    "lms_activity": np.round(lms_activity, 1),
    "previous_gpa": np.round(previous_gpa, 2),
    "commute_minutes": np.round(commute, 1),
    "at_risk": at_risk,
})

print("Master dataset shape:", df_master.shape)
print("\nTarget balance:")
display(df_master["at_risk"].value_counts().rename(index={0:"not_at_risk", 1:"at_risk"}))

## Task 3.1 — Personalized Dataset

Enter the last four digits of your student ID.

Your ID determines a reproducible working sample.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 8000 + (STUDENT_ID_LAST4 % 2000)

df = df_master.sample(
    n=720,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your tree seed:", SEED)
print("Working dataset shape:", df.shape)

In [ ]:
feature_cols = [
    "attendance_pct",
    "quiz_score",
    "missed_labs",
    "lms_activity",
    "previous_gpa",
    "commute_minutes",
]

X = df[feature_cols]
y = df["at_risk"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

## Task 3.2 — Predict Before Training

Before fitting the tree:

1. Which feature do you expect to appear near the root?
2. Which feature pair might form an interaction?
3. Do you expect a shallow or deep tree to fit training data better?
4. Do trees require standardization for threshold-based splits? Explain.

In [ ]:
tree_default = DecisionTreeClassifier(
    criterion="entropy",
    random_state=SEED
)

tree_default.fit(X_train, y_train)

train_pred = tree_default.predict(X_train)
valid_pred = tree_default.predict(X_valid)

print("Default tree depth:", tree_default.get_depth())
print("Default tree leaves:", tree_default.get_n_leaves())
print("Training accuracy:", round(accuracy_score(y_train, train_pred), 4))
print("Validation accuracy:", round(accuracy_score(y_valid, valid_pred), 4))

## Task 3.3 — Interpret the First Result

Answer:

1. Is training accuracy extremely high?
2. Is validation accuracy lower?
3. What does a large gap suggest?
4. Why can an unrestricted tree memorize training-specific details?

# Part IV — Visualize and Read the Tree

A small tree is easier to interpret than an unrestricted one.

We will fit a shallow tree with:

$$
max\_depth=3.
$$

In [ ]:
tree_small = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    min_samples_leaf=8,
    random_state=SEED
)

tree_small.fit(X_train, y_train)

plt.figure(figsize=(18, 9))
plot_tree(
    tree_small,
    feature_names=feature_cols,
    class_names=["Not Risk", "At Risk"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree — max_depth=3")
plt.show()

## Task 4.1 — Read the Root

From the plotted tree, record:

- root feature;
- root threshold;
- number of samples at the root;
- root entropy;
- root class distribution.

Then explain:

> Why does the root split have strong influence over the entire tree?

In [ ]:
print(export_text(
    tree_small,
    feature_names=feature_cols
))

## Task 4.2 — Convert One Path into a Rule

Choose one root-to-leaf path and rewrite it as:

```text
IF ...
AND ...
THEN predict ...
```

Then answer:

1. Is the rule easy to explain to a non-technical stakeholder?
2. Does interpretability guarantee the rule is causally correct?

# Part V — Trace One Prediction

A decision tree prediction can be traced through the decision path.

In [ ]:
sample_position = SEED % len(X_valid)
sample_index = X_valid.index[sample_position]
sample_row = X_valid.loc[[sample_index]]

print("Selected validation student:")
display(sample_row)
print("True label:", int(y_valid.loc[sample_index]))
print("Predicted class:", int(tree_small.predict(sample_row)[0]))
print("Predicted probabilities:", np.round(tree_small.predict_proba(sample_row)[0], 4))

In [ ]:
node_indicator = tree_small.decision_path(sample_row)
leaf_id = tree_small.apply(sample_row)[0]

feature = tree_small.tree_.feature
threshold = tree_small.tree_.threshold

node_ids = node_indicator.indices[
    node_indicator.indptr[0]:
    node_indicator.indptr[1]
]

print("Decision path:")

for node_id in node_ids:
    if node_id == leaf_id:
        print(f"  Node {node_id}: LEAF")
        continue

    feature_name = feature_cols[feature[node_id]]
    value = sample_row.iloc[0, feature[node_id]]
    thresh = threshold[node_id]
    direction = "<=" if value <= thresh else ">"

    print(
        f"  Node {node_id}: "
        f"{feature_name} = {value:.3f} "
        f"{direction} {thresh:.3f}"
    )

## Task 5.1 — Explain the Prediction

For your assigned student:

1. list the split decisions in order;
2. identify the final leaf;
3. state the predicted class;
4. state the predicted probability;
5. explain whether the prediction seems reasonable from the feature values.

This explanation should be possible without saying “because the library predicted it.”

# Part VI — Tree Depth and Overfitting

We now vary:

$$
max\_depth\in\{1,2,3,4,5,6,8,10,None\}.
$$

Prediction before running:

- training accuracy should usually increase with depth;
- validation accuracy may improve and then worsen;
- very deep trees may overfit.

In [ ]:
depth_values = [1, 2, 3, 4, 5, 6, 8, 10, None]
depth_rows = []
depth_models = {}

for depth in depth_values:
    model = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=depth,
        random_state=SEED
    )
    model.fit(X_train, y_train)

    train_p = model.predict(X_train)
    valid_p = model.predict(X_valid)

    depth_rows.append({
        "max_depth": "None" if depth is None else depth,
        "actual_depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "train_accuracy": accuracy_score(y_train, train_p),
        "valid_accuracy": accuracy_score(y_valid, valid_p),
        "valid_f1": f1_score(y_valid, valid_p),
    })

    depth_models[depth] = model

depth_table = pd.DataFrame(depth_rows)
display(depth_table.round(4))

In [ ]:
plot_df = depth_table.copy()
plot_df["depth_numeric"] = [
    row["actual_depth"]
    for _, row in plot_df.iterrows()
]

plt.figure(figsize=(8, 5))
plt.plot(
    plot_df["depth_numeric"],
    plot_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)
plt.plot(
    plot_df["depth_numeric"],
    plot_df["valid_accuracy"],
    marker="o",
    label="Validation accuracy"
)
plt.xlabel("Actual Tree Depth")
plt.ylabel("Accuracy")
plt.title("Tree Depth: Training vs. Validation")
plt.legend()
plt.show()

## Task 6.1 — Diagnose Underfitting and Overfitting

Answer:

1. Which depth gives the highest training accuracy?
2. Which depth gives the highest validation accuracy?
3. Which shallow depths appear to underfit?
4. Which deep settings appear to overfit?
5. Why is training accuracy alone a poor way to choose depth?

# Part VII — `min_samples_leaf` as Pre-Pruning

Another way to control complexity is to prevent very small leaves.

We will test:

$$
min\_samples\_leaf\in\{1,2,5,10,20,30\}.
$$

## Task 7.1 — Predict Before Running

As `min_samples_leaf` increases, predict what happens to:

- number of leaves;
- training accuracy;
- validation stability;
- interpretability.

In [ ]:
leaf_values = [1, 2, 5, 10, 20, 30]
leaf_rows = []

for leaf_size in leaf_values:
    model = DecisionTreeClassifier(
        criterion="entropy",
        min_samples_leaf=leaf_size,
        random_state=SEED
    )
    model.fit(X_train, y_train)

    train_p = model.predict(X_train)
    valid_p = model.predict(X_valid)

    leaf_rows.append({
        "min_samples_leaf": leaf_size,
        "depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "train_accuracy": accuracy_score(y_train, train_p),
        "valid_accuracy": accuracy_score(y_valid, valid_p),
        "valid_f1": f1_score(y_valid, valid_p),
    })

leaf_table = pd.DataFrame(leaf_rows)
display(leaf_table.round(4))

## Task 7.2 — Interpret Pre-Pruning

Answer:

1. Which leaf-size setting has the best validation $F_1$?
2. What happens when leaf size becomes too large?
3. Why can larger leaves reduce variance?
4. Why can overly large leaves increase bias?

# Part VIII — Cost-Complexity Pruning

Scikit-learn supports post-pruning using the cost-complexity parameter:

```text
ccp_alpha
```

Larger `ccp_alpha` values penalize tree complexity more strongly.

We will obtain candidate pruning strengths from the training data.

In [ ]:
pruning_tree = DecisionTreeClassifier(
    criterion="entropy",
    random_state=SEED
)

path = pruning_tree.cost_complexity_pruning_path(
    X_train,
    y_train
)

ccp_alphas = path.ccp_alphas

# Use a small representative subset of candidates for the lab.
if len(ccp_alphas) > 8:
    indices = np.linspace(
        0,
        len(ccp_alphas) - 1,
        8,
        dtype=int
    )
    alpha_candidates = np.unique(ccp_alphas[indices])
else:
    alpha_candidates = np.unique(ccp_alphas)

alpha_rows = []
alpha_models = {}

for alpha in alpha_candidates:
    model = DecisionTreeClassifier(
        criterion="entropy",
        ccp_alpha=float(alpha),
        random_state=SEED
    )

    model.fit(X_train, y_train)

    train_p = model.predict(X_train)
    valid_p = model.predict(X_valid)

    alpha_rows.append({
        "ccp_alpha": float(alpha),
        "depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "train_accuracy": accuracy_score(y_train, train_p),
        "valid_accuracy": accuracy_score(y_valid, valid_p),
        "valid_f1": f1_score(y_valid, valid_p),
    })

    alpha_models[float(alpha)] = model

alpha_table = pd.DataFrame(alpha_rows)
display(alpha_table.round(5))

## Task 8.1 — Interpret Post-Pruning

Answer:

1. As `ccp_alpha` increases, what happens to depth and number of leaves?
2. Does training accuracy generally increase or decrease?
3. Is there an intermediate pruning level with stronger validation performance?
4. Why can removing some branches improve generalization?
5. Why should `ccp_alpha` be selected with validation evidence?

# Part IX — Feature Importance

Decision trees can report impurity-based feature importance.

This measures how much each feature contributes to impurity reduction across the fitted tree.

It does **not** prove causality.

In [ ]:
best_depth_value = depth_values[
    int(np.argmax([
        row["valid_f1"] for row in depth_rows
    ]))
]

best_depth_model = depth_models[best_depth_value]

importance_table = pd.DataFrame({
    "feature": feature_cols,
    "importance": best_depth_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance_table.round(4))

## Task 9.1 — Interpret Feature Importance Carefully

Answer:

1. Which feature has highest importance?
2. Does that match your root-split prediction?
3. Does high feature importance prove the feature causes risk?
4. Why might importance change if the training sample changes?
5. Why can correlated features share or compete for importance?

# Part X — Validation-Based Model Selection

We will compare three candidates:

1. unrestricted tree;
2. best depth by validation $F_1$;
3. best cost-complexity-pruned tree by validation $F_1$.

The test set remains untouched.

In [ ]:
best_depth_idx = depth_table["valid_f1"].idxmax()
best_depth_label = depth_table.loc[best_depth_idx, "max_depth"]
best_depth_actual = int(depth_table.loc[best_depth_idx, "actual_depth"])

if best_depth_label == "None":
    best_depth_key = None
else:
    best_depth_key = int(best_depth_label)

best_depth_model = depth_models[best_depth_key]

best_alpha_idx = alpha_table["valid_f1"].idxmax()
best_alpha = float(alpha_table.loc[best_alpha_idx, "ccp_alpha"])
best_alpha_model = alpha_models[best_alpha]

candidate_models = {
    "Unrestricted": tree_default,
    f"Best depth ({best_depth_label})": best_depth_model,
    f"Best ccp_alpha ({best_alpha:.5f})": best_alpha_model,
}

selection_rows = []

for name, model in candidate_models.items():
    pred = model.predict(X_valid)

    selection_rows.append({
        "candidate": name,
        "depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "accuracy": accuracy_score(y_valid, pred),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "f1": f1_score(y_valid, pred, zero_division=0),
    })

selection_table = pd.DataFrame(selection_rows)
display(selection_table.round(4))

## Task 10.1 — Choose the Final Tree

Use validation evidence to select a final model.

Your answer must mention:

- validation $F_1$;
- tree depth;
- number of leaves;
- interpretability;
- overfitting risk.

If two models have very similar $F_1$, explain whether you would prefer the simpler tree.

# Part XI — Personalized Tree Challenge

Your student ID assigns one `max_depth` value:

$$
2,\;3,\;4,\;5,\;7,\;9.
$$

You must predict its behavior before fitting.

In [ ]:
personal_depths = [2, 3, 4, 5, 7, 9]
PERSONAL_DEPTH = personal_depths[
    SEED % len(personal_depths)
]

print("Your assigned max_depth:", PERSONAL_DEPTH)

## Task 11.1 — Predict Before Fitting

Predict:

1. whether your tree will underfit, fit reasonably, or overfit;
2. whether training accuracy will be higher/lower than the depth-3 tree;
3. whether validation $F_1$ is guaranteed to improve;
4. whether the final tree will be easy to interpret.

In [ ]:
personal_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=PERSONAL_DEPTH,
    random_state=SEED
)

personal_tree.fit(X_train, y_train)

personal_train_pred = personal_tree.predict(X_train)
personal_valid_pred = personal_tree.predict(X_valid)

personal_summary = {
    "depth": personal_tree.get_depth(),
    "leaves": personal_tree.get_n_leaves(),
    "train_accuracy": accuracy_score(y_train, personal_train_pred),
    "valid_accuracy": accuracy_score(y_valid, personal_valid_pred),
    "valid_f1": f1_score(y_valid, personal_valid_pred),
}

print(personal_summary)

## Task 11.2 — Analyze Your Personalized Tree

Answer:

1. Was your prediction correct?
2. Did training accuracy rise as expected?
3. Did validation $F_1$ improve?
4. How many leaves were created?
5. Would you recommend this tree? Why or why not?

# Part XII — Deliberate Debugging

Consider:

```python
tree = DecisionTreeClassifier(max_depth=None)
tree.fit(X_train, y_train)

while tree.score(X_train, y_train) < 1.0:
    increase_complexity()
```

This logic assumes perfect training accuracy is the goal.

Explain why that is dangerous.

## Task 12.1 — Diagnose the Overfitting Logic

Answer:

1. Why can perfect training accuracy be misleading?
2. What should guide complexity selection instead?
3. Which hyperparameters can control tree complexity?
4. What pattern in train/validation accuracy suggests overfitting?

## Task 12.2 — Fix an Entropy Bug

The following function is wrong:

```python
def bad_entropy(p):
    return p * np.log2(p)
```

Explain what is missing, then complete a correct binary-probability entropy function.

In [ ]:
def entropy_from_probability(p):
    p = float(p)

    # TODO:
    # Return entropy for probabilities p and (1-p).
    # Handle p=0 and p=1 safely.
    return None

In [ ]:
assert abs(entropy_from_probability(0.5) - 1.0) < 1e-12
assert abs(entropy_from_probability(1.0) - 0.0) < 1e-12
assert abs(entropy_from_probability(0.0) - 0.0) < 1e-12

print("Entropy debugging test passed.")

# Part XIII — Final Test Evaluation

After the final tree structure has been selected from validation evidence, evaluate once on the test set.

The code below selects the candidate with highest validation $F_1$.

In [ ]:
winner_name = selection_table.loc[
    selection_table["f1"].idxmax(),
    "candidate"
]

winner_model = candidate_models[winner_name]

test_pred = winner_model.predict(X_test)

test_cm = confusion_matrix(y_test, test_pred)
tn, fp, fn, tp = test_cm.ravel()

test_results = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
}

print("Selected model:", winner_name)
print("Depth:", winner_model.get_depth())
print("Leaves:", winner_model.get_n_leaves())

print("\nTest confusion matrix:")
display(pd.DataFrame(
    test_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

print("\nFinal test metrics:")
display(pd.Series(test_results).round(4).to_frame("value"))

## Task 13.1 — Final Generalization Statement

Write 4–6 sentences that include:

- selected tree;
- why it was selected;
- depth and number of leaves;
- test accuracy;
- test recall;
- test $F_1$;
- whether test behavior is consistent with validation;
- one limitation of the tree.

Do not change tree complexity after seeing the test result.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. Explain entropy using one mixed and one pure node.
2. What is information gain?
3. Why does a tree choose one split over another?
4. Why can an unrestricted tree overfit?
5. What does `max_depth` control?
6. What does `min_samples_leaf` control?
7. How can cost-complexity pruning improve generalization?
8. Trace the prediction path for your assigned validation student.
9. Why does feature importance not imply causality?
10. For your personalized depth, explain its train/validation behavior.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What did manually calculating information gain help you understand?
2. Why is a pure training tree not necessarily desirable?
3. Which complexity-control method was easiest to interpret?
4. Why can a smaller tree be preferable even if validation performance is almost the same?
5. What is one major difference between tree learning and logistic-regression learning?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] completed entropy function;
- [ ] completed Gini function;
- [ ] completed weighted-entropy function;
- [ ] completed information-gain function;
- [ ] manual root-split interpretation;
- [ ] your own student-ID-derived dataset;
- [ ] unrestricted-tree analysis;
- [ ] shallow-tree visualization;
- [ ] one prediction-path explanation;
- [ ] depth experiment;
- [ ] `min_samples_leaf` experiment;
- [ ] cost-complexity pruning experiment;
- [ ] feature-importance interpretation;
- [ ] validation-based final model selection;
- [ ] personalized depth challenge;
- [ ] corrected entropy debugging task;
- [ ] final test evaluation;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\boxed{
\text{impurity}
\rightarrow
\text{split quality}
\rightarrow
\text{recursive structure}
\rightarrow
\text{complexity}
\rightarrow
\text{generalization}
}
$$

# Lab 8 Summary

You should now be able to explain how a decision tree learns.

### Entropy

$$
H(S)
=
-\sum_k p_k\log_2 p_k
$$

### Information gain

$$
IG
=
H(parent)
-
\sum_j
\frac{|child_j|}{|parent|}
H(child_j)
$$

### Main lessons

- trees learn by recursively choosing impurity-reducing splits;
- pure child nodes reduce uncertainty;
- unrestricted trees can memorize training data;
- `max_depth` and `min_samples_leaf` provide pre-pruning;
- `ccp_alpha` supports post-pruning;
- validation data should control tree complexity;
- smaller trees are often easier to interpret and may generalize better.

**Next lab:** K-Nearest Neighbors and the Effect of Representation.